In [1]:
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
from tnn_mdls.sim_utils import *
import time
from veriloggen import *
import copy
from model import Model
import numpy as np
import random
from layer import Layer
from utils import *

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt

In [3]:
mnist_posneg = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),PosNeg(0.5),transforms.CenterCrop(3)]))

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|███████████████████████████████████████████████████████████████████| 9912422/9912422 [00:04<00:00, 2009456.40it/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|████████████████████████████████████████████████████████████████████████| 28881/28881 [00:00<00:00, 415545.59it/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|███████████████████████████████████████████████████████████████████| 1648877/1648877 [00:00<00:00, 2024832.22it/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████████████████████████████████████████████████████████████████████| 4542/4542 [00:00<00:00, 601285.51it/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



In [4]:
mnist_img = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(), transforms.CenterCrop(3)]))

In [ ]:
img, label = mnist_img[0]
plt.imshow(img.squeeze(), cmap="gray")

In [ ]:
img, label = mnist_posneg[0]
img.shape

In [168]:
def MNIST_single_column(num_synapse=18, thres=11, tres=1, wres=4, wave=10, verbose=False):
    f = open("MNIST_single_column", "w")
    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        img, label = mnist_posneg[w]
        img_flat = img.flatten()
        img_flat += 1
        # generate input spike times
        img_flat[img_flat==float('inf')] = -1
        spike_times = np.array(img_flat)
        reset_times = np.zeros(spike_times.shape)
        for i in range(num_synapse):
            if spike_times[i]>0:
                reset_times[i] = spike_times[i]+(2**wres-1)-(2**tres-1)
            else:
                reset_times[i] = -1

        # t-window
        if verbose:
            print('t-window')
        in_bits = ['0'] * num_synapse
        delays = []
        layer_in = [0]
        delay = 0

        for t in range(2**tres):
            if t in spike_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Generate input spike at time t
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]

            delay+=1

        layer_in += [int(layer_in_string, 2)]
        delays += [delay]

        # w-window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres):
            if t in reset_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]

            delay+=1

        delays+=[delay]

        if verbose:
            print(layer_in, delays)
        # Write loop
        for i in range(len(layer_in)):
            f.write('layer_in('+str(layer_in[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
    f.write('layer_in(0),\n')
    f.write('Delay(100),\n')

In [170]:
MNIST_single_column(num_synapse=18, thres=30, tres=1, wres=3, wave=20, verbose=False)

In [158]:
m = Module('test_simple')

# parameters
num_col = 1
num_neurons = 10
num_synapse = 18
tres = 2
wres = 4
thres = 11

# Initial inputs
layer_in = 0
w_init = 0
capture_brv = -1
minus_brv = -1
search_brv = -1
backoff_brv = -1
min_brv = -1
F_brv = -1

L = Layer(layer_type="Simple", num_col=num_col, num_neurons=num_neurons, p_dist=num_synapse, wres_dist=wres, thres=thres)
simple = L.Simple_Layer(0)
dut = Submodule(m, simple, 'dut')

# Setup waveform dump and simulation environment
dump, clk, grst, rstb = init_dump(dut, m)

input_init = [[0], # layer_in
              [0], # w_init
              [capture_brv], # capture_brv
              [minus_brv], # minus_brv
              [search_brv], # search_brv
              [backoff_brv], # backoff_brv
              [min_brv], # min_brv
              [F_brv]] # F_brv
delay_init = [0]

add_to_dump(dump, dut, input_init, delay_init, 1)

rstb_gen(dump, rstb, 8)
grst_gen(m, ((2**tres)+(2**wres)))

add_to_dump_from_file(dump, dut, "tnn_mdls/tests/MNIST_single_column")

In [159]:
gen_file, rtl_path = gen_verilog(module = m, filename = 'mnist_tb.sv')

In [6]:
# Trying to generate inputs randomly

In [15]:
def neuronbody_test(ip_size=4, thres=13, tres=4, wres=4, wave=10, verbose=False):
    f = open("neuronbody_test_random", "w")
    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        
        # Generate spike times
        spike_times = []
        for i in range(ip_size):
            spike_times += [random.randint(1, (2**tres)-1)]
        spike_times = np.array(spike_times)
        reset_times = spike_times+(2**wres-1)-(2**tres-1)
        
        if verbose:
            print('spike_times: ', spike_times)
        
        # tres window
        if verbose:
            print('t-window')
            
        in_bits = ['0'] * ip_size
        delays = []
        acc_ins = [0]
        delay = 0
        for t in range(2**tres):
            if t in spike_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Flip bits of time spikes
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate acc_in
                acc_string = ''
                for i in range(ip_size):
                    acc_string += in_bits[i]
                acc_ins += [int(acc_string, 2)]
            
            if verbose:
                print(in_bits)

            delay+=1

        acc_ins += [int(acc_string, 2)]
        delays+=[delay]
        
        # wres window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres):
            if t in reset_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate acc_in
                acc_string = ''
                for i in range(ip_size):
                    acc_string += in_bits[i]
                acc_ins += [int(acc_string, 2)]
                
            if verbose:
                print(in_bits)

            delay+=1
        delays+=[delay]
        
        if verbose:
            print(acc_ins, delays)
        
        # Write loop
        for i in range(len(acc_ins)):
            f.write('acc_in('+str(acc_ins[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
    f.close()
    
def stdp_test(tres=3, wres=4, input_prob=0.75, output_prob=0.75, wave=10, verbose=False):
    f = open("stdp_test_random", "w")
    for w in range(wave):
        
        f.write('# Wave: '+ str(w) +'\n')
        
        weight_in = random.randint(0, (2**wres-1))
        f.write('weight_in('+str(weight_in)+'),\n')
        
        e_in = 0
        e_out = 0
        delay = 0
        e_ins = [0]
        e_outs = [0]
        delays = []
        if_input_spike = np.random.choice([0,1], p=[1-input_prob, input_prob])
        if_output_spike = np.random.choice([0,1], p=[1-output_prob, output_prob])

        # Generate spike times
        if if_input_spike:
            e_in_time = random.randint(1, (2**tres)-1)
        else:
            e_in_time = -1
            e_in_reset = -1
        if if_output_spike:
            e_out_time = random.randint(1, (2**tres))
        else:
            e_out_time = -1
            e_out_reset = -1

        #print(e_in_time, e_out_time)

        # t-window
        for t in range(2**tres):
            if e_in_time == t:
                e_in = 1
                e_ins += [1]
                e_outs += [e_out]
                delays += [delay]
                delay = 0
            if e_out_time == t:
                e_out = 1
                e_outs += [1]
                e_ins += [e_in]
                delays += [delay]
                delay = 0
            delay += 1

        # w-window
        e_ins += [e_in]
        e_outs += [e_out]
        delays += [delay+2**wres]
        
        if verbose:
            print('delays:', delays)
            print('ein:', e_ins)
            print('eout:', e_outs)
            print('weight_in: ', weight_in)
        
        # Write loop
        for i in range(len(delays)):
            f.write('ein('+str(e_ins[i])+'),\n')
            f.write('eout('+str(e_outs[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
            
def fsm_synapse_test(w_init=5, tres=4, wres=4, wave=10, verbose=False):    
    f = open("tnn_mdls/tests/fsm_synapse_test", "w")
    
    # set w_init
    f.write('w_init(str(w_init)),\n\n')
    
    # inc test
    f.write('# Increment tests\n\n')
    
    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        f.write('dec(0),\n')
        f.write('inc(0),\n')
        
        input_spike_time = random.randint(1, (2**tres)-1)
        total_wave_time = 2**tres + 2**wres - 1

        f.write('Delay('+str(input_spike_time)+'),\n')
        f.write('input_spike(1),\n')
        f.write('inc(1),\n')
        f.write('Delay('+str((2**wres))+'),\n')
        f.write('input_spike(0),\n')
        total_wave_time -= input_spike_time+(2**wres)
        f.write('Delay('+str(total_wave_time)+'),\n')
        f.write('\n')
        
    # dec test
    f.write('# Decrement tests\n\n')

    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        f.write('dec(0),\n')
        f.write('inc(0),\n')
        
        input_spike_time = random.randint(1, (2**tres)-1)
        inc = random.randint(0,1)
        total_wave_time = 2**tres + 2**wres - 1

        f.write('Delay('+str(input_spike_time)+'),\n')
        f.write('input_spike(1),\n')
        f.write('dec(1),\n')
        f.write('Delay('+str((2**wres))+'),\n')
        f.write('input_spike(0),\n')
        total_wave_time -= input_spike_time+(2**wres)
        f.write('Delay('+str(total_wave_time)+'),\n')
        f.write('\n')
        
    f.write('input_spike(0),\n')
    f.write('Delay(100),\n')

def wta_test(Q=4, tres=4, wres=4, wave=10, verbose=False):
    f = open("wta_test_random", "w")
    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        
        # Generate spike times
        spike_times = []
        for i in range(Q):
            spike_times += [random.randint(1, (2**tres)-1)]
        spike_times = np.array(spike_times)
        reset_times = spike_times+(2**wres-1)-(2**tres-1)
        
        if verbose:
            print('spike_times: ', spike_times)
        
        # tres window
        if verbose:
            print('t-window')
            
        in_bits = ['0'] * Q
        delays = []
        ec_spikes = [0]
        delay = 0
        for t in range(2**tres):
            if t in spike_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Flip bits of time spikes
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate ec_spike
                ec_string = ''
                for i in range(Q):
                    ec_string += in_bits[i]
                ec_spikes += [int(ec_string, 2)]
            
            if verbose:
                print(in_bits)

            delay+=1

        ec_spikes += [int(ec_string, 2)]
        delays+=[delay]
        
        # wres window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres):
            if t in reset_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate ec_spike
                ec_string = ''
                for i in range(Q):
                    ec_string += in_bits[i]
                ec_spikes += [int(ec_string, 2)]
                
            if verbose:
                print(in_bits)

            delay+=1
        delays+=[delay]
        
        if verbose:
            print(ec_spikes, delays)
        
        # Write loop
        for i in range(len(ec_spikes)):
            f.write('ec_spikes('+str(ec_spikes[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
    f.close()
    
def segment_test(ip_size_dist=16, ip_size_prox=1, thres=13, tres=4, wres=4, wave=10, verbose=False):
    f = open("segment_test_random", "w")
    cat_ip_size = ip_size_dist + ip_size_prox
    f.write('# Set w_inits:\n')
    f.write('w_init_dist(5),\n')
    f.write('w_init_prox(5),\n\n')
    # Test 4 different inc and dec
    for n in range(4):
        if n == 0:
            f.write('# Test inc dist:\n')
            f.write('inc_dist(' + str(2**ip_size_dist-1) + '),\n')
            f.write('inc_prox(0),\n')
            f.write('dec_dist(0),\n')
            f.write('dec_prox(0),\n\n')
        elif n == 1:
            f.write('# Test inc prox:\n')
            f.write('inc_dist(0),\n')
            f.write('inc_prox(' + str(2**ip_size_prox-1) + '),\n')
            f.write('dec_dist(0),\n')
            f.write('dec_prox(0),\n\n')
        elif n == 2:
            f.write('# Test dec dist:\n')
            f.write('inc_dist(0),\n')
            f.write('inc_prox(0),\n')
            f.write('dec_dist(' + str(2**ip_size_dist-1) + '),\n')
            f.write('dec_prox(0),\n\n')
        elif n == 3:
            f.write('# Test dec prox:\n')
            f.write('inc_dist(0),\n')
            f.write('inc_prox(0),\n')
            f.write('dec_dist(0),\n')
            f.write('dec_prox(' + str(2**ip_size_prox-1) + '),\n\n')
            
        for w in range(wave):
            f.write('# Wave: '+ str(w) +'\n')

            # Generate spike times
            spike_times = []
            for i in range(cat_ip_size):
                spike_times += [random.randint(1, (2**tres)-1)]
            spike_times = np.array(spike_times)
            reset_times = spike_times+(2**wres-1)-(2**tres-1)

            if verbose:
                print('spike_times: ', spike_times)

            # tres window
            if verbose:
                print('t-window')

            in_bits = ['0'] * cat_ip_size
            delays = []
            input_spikes = [0]
            delay = 0
            for t in range(2**tres):
                if t in spike_times:
                    delays += [delay]
                    delay = 0
                    time_matches = np.where(spike_times == t)[0]

                    # Flip bits of time spikes
                    for i in range(len(time_matches)):
                        index = time_matches[i]
                        in_bits[index] = '1'

                    # Calculate input_spike
                    in_string = ''
                    for i in range(cat_ip_size):
                        in_string += in_bits[i]
                    input_spikes += [int(in_string, 2)]

                if verbose:
                    print(in_bits)

                delay+=1

            input_spikes += [int(in_string, 2)]
            delays+=[delay]

            # wres window
            if verbose:
                print('w-window')
            delay = 0
            for t in range(2**wres):
                if t in reset_times:
                    delays += [delay]
                    delay = 0
                    time_matches = np.where(reset_times == t)[0]
                    for i in range(len(time_matches)):
                        index = time_matches[i]
                        in_bits[index] = '0'

                    # Calculate input_spike
                    in_string = ''
                    for i in range(cat_ip_size):
                        in_string += in_bits[i]
                    input_spikes += [int(in_string, 2)]

                if verbose:
                    print(in_bits)

                delay+=1
            delays+=[delay]

            if verbose:
                print(input_spikes, delays)

            # Write loop
            input_format = '0'+str(cat_ip_size)+'b'
            for i in range(len(input_spikes)):
                dist_spikes = format(input_spikes[i], input_format)[ip_size_prox:ip_size_prox+ip_size_dist]
                prox_spikes = format(input_spikes[i], input_format)[0:ip_size_prox]
                f.write('input_spikes_dist('+str(int(dist_spikes, 2))+'),\n')
                f.write('input_spikes_prox('+str(int(prox_spikes, 2))+'),\n')
                f.write('Delay('+str(delays[i])+'),\n')
                f.write('\n')
    f.close()

In [16]:
#random.seed(14)
#wta_test(Q=4, tres=4, wres=4, wave=10, verbose=False)
#stdp_test(tres=3, wres=4, input_prob=0.75, output_prob=0.75, wave=10, verbose=False)
fsm_synapse_test(w_init=0, tres=1, wres=3, wave=30, verbose=False)
#neuronbody_test(ip_size=4, thres=13, tres=1, wres=3, wave=5, verbose=True)
#segment_test(ip_size_dist=4, ip_size_prox=1, thres=13, tres=3, wres=4, wave=5, verbose=False)

In [150]:
def extract_number(line):
    result = ""
    for c in line:
        if c.isnumeric():
            result+=c
    return int(result)

In [152]:
# Take (randomly generated) files and parse through them to extract inputs and delays
ip_size=4
thres=13
wres=4
wave=20
f = open("neuronbody_test_random", "r")
delays = []
input_names = ['acc_in']
num_waves = 20
inputs = np.zeros((len(input_names), num_waves))
inputs_mask = np.zeros((len(input_names), num_waves))
current_wave = 0

while True:
    line = f.readline()
    if not line:
        break
    else:
        line = line.rstrip()
        if line.startswith('Delay'):
            #print(line)
            delays += [extract_number(line)]
        elif line.startswith('# Wave'):
            #print(line)
            current_wave = extract_number(line)
        elif line and not line.startswith('#'):
            #print(line)
            input_index = input_names.index(line.split('(')[0])
            inputs[input_index][current_wave] = extract_number(line)
            inputs_mask[input_index][current_wave] = 1

In [153]:
# Apply mask to remove invalid inputs not in waves
masked_inputs = np.zeros((len(input_names), num_waves))
for i in range(inputs.shape[0]):
    for j in range(inputs.shape[1]):
        if inputs_mask[i][j] == 1:
            masked_inputs[i][j] = inputs[i][j]
        elif inputs_mask[i][j] == 0 and j!=0:
            masked_inputs[i][j] = masked_inputs[i][j-1]
masked_inputs

array([[ 9., 11.,  1.,  9., 11.,  6.,  8.,  2., 14.,  3.,  2., 13., 12.,
        12., 15.,  5.,  6.,  3., 10.,  1.]])

In [154]:
# Generate input matrix for every cycle
delays = np.array(delays)
delay_cumsum = np.cumsum(delays)
delay_cumsum = np.insert(delay_cumsum, 0, 0)

total_sim_time = sum(delays)
input_matrix = np.zeros((inputs.shape[0], total_sim_time))

for t in range(total_sim_time):
    if t in delay_cumsum:
        for i in range(inputs.shape[0]):
            input_matrix[i][t] = masked_inputs[i][np.where(delay_cumsum==t)[0][0]]
    elif t != 0:
        for i in range(inputs.shape[0]):
            input_matrix[i][t] = input_matrix[i][t-1]
input_matrix

array([[ 9.,  9.,  9.,  9., 11.,  1.,  1.,  9.,  9.,  9., 11., 11., 11.,
         6.,  6.,  6.,  8.,  8.,  8.,  2.,  2.,  2., 14., 14., 14.,  3.,
         2.,  2., 13., 13., 13., 13., 12., 12., 12., 12., 12., 12., 15.,
         5.,  5.,  5.,  5.,  6.,  6.,  6.,  6.,  3.,  3.,  3., 10.,  1.,
         1.,  1.]])

In [159]:
# Compute expected body_pot and output spikes
pot = 0
grst_period = 24
acc_in = input_matrix[0]

for t in range(total_sim_time):
    if t==grst_period:
        pot = 0
    else:
        pot += bin(int(acc_in[t])).count("1")
    print(pot)
    if pot >= thres:
        print('spike at t=', t)
        pot = 0

2
4
6
8
11
12
13
spike at t= 6
2
4
6
9
12
15
spike at t= 12
2
4
6
7
8
9
10
11
12
15
spike at t= 22
3
0
2
3
4
7
10
13
spike at t= 30
3
5
7
9
11
13
spike at t= 36
2
6
8
10
12
14
spike at t= 42
2
4
6
8
10
12
14
spike at t= 49
2
3
4
5
